<a href="https://colab.research.google.com/github/degus349-stack/signature-vectorizer/blob/branch01/Signature-Vectorizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kelompok D

- Nabila Zahra (2401010593)
- Zahwa Pahrisa Rahman (2401010634)
- Devin Dilan Johones (2401010591)
- Justinus Marselino Kabosu (2401010614)
- I Made Pande Wiradana (2401010616)



# FINAL CHAPTER

## Install & Import

In [109]:
!pip install opencv-python-headless scikit-image matplotlib svgwrite gradio pandas -q

import cv2
import gradio as gr
import numpy as np
import svgwrite
import zipfile
import pandas as pd
import io
import os
import tempfile

from skimage import morphology
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

## CSS

In [110]:
CUSTOM_CSS = """
:root {
    --bg-main:    #0f1117;
    --bg-card:    #1a1d27;
    --bg-input:   #12151f;
    --border:     #2a2d3a;

    --accent:     #7c6fcd;
    --accent-dim: #3d3670;

    --text-main:  #e8e8f0;
    --text-muted: #9a9ab5;
    --text-label: #b3b3c7;

    --radius:     12px;
}

body,
.gradio-container {
    background: var(--bg-main) !important;
    color: var(--text-main) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}

/* ==========================================================
   HEADER
========================================================== */

.app-header {
    text-align: center;
    padding: 24px 20px;
    margin-bottom: 18px;
    border-bottom: 1px solid var(--border);
}

.app-header h1 {
    margin: 0;
    font-size: 28px;
    font-weight: 700;
    color: var(--text-main);
}

.app-header p {
    margin-top: 8px;
    color: var(--text-muted);
    font-size: 14px;
    line-height: 1.5;
}

/* ==========================================================
   CARD
========================================================== */

.card {
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 18px;
    margin-bottom: 16px;
}

.final-card {
    background: linear-gradient(
        180deg,
        var(--bg-card),
        #161926
    );
    border: 1px solid var(--accent-dim);
}

/* ==========================================================
   STEP
========================================================== */

.step-badge {
    display: inline-flex;
    align-items: center;

    background: var(--accent-dim);
    color: #d8d0ff;

    font-size: 11px;
    font-weight: 700;

    padding: 5px 12px;
    border-radius: 20px;

    margin-bottom: 10px;
}

.step-title {
    font-size: 16px;
    font-weight: 700;
    color: var(--text-main);
    margin-bottom: 6px;
}

.step-desc {
    font-size: 13px;
    color: var(--text-muted);
    line-height: 1.6;
}

/* ==========================================================
   SECTION TITLE
========================================================== */

.section-label {
    font-size: 13px;
    font-weight: 700;
    letter-spacing: 1px;
    text-transform: uppercase;

    color: var(--accent);

    margin: 14px 0 10px;
}

/* ==========================================================
   DIAGNOSIS BOX
========================================================== */

.warn-box {
    background: #2a1f0e;
    border: 1px solid #7a4f1a;
    color: #f5c97a;

    border-radius: 10px;
    padding: 14px 16px;
    line-height: 1.7;
}

.ok-box {
    background: #0e2a1a;
    border: 1px solid #1a6640;
    color: #6ee7a8;

    border-radius: 10px;
    padding: 14px 16px;
    line-height: 1.7;
}

.pipeline-box {
    background: #0e1a2a;
    border: 1px solid #1a4f7a;
    color: #7ac4f5;

    border-radius: 10px;
    padding: 14px 16px;
    line-height: 1.7;
}

/* ==========================================================
   METRICS
========================================================== */

.diag-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);

    gap: 12px;
    margin-top: 12px;
    margin-bottom: 12px;
}

.diag-metric {
    background: var(--bg-input);
    border: 1px solid var(--border);

    border-radius: 10px;
    padding: 12px;

    text-align: center;
}

.diag-metric .val {
    font-size: 18px;
    font-weight: 700;
    color: var(--text-main);
}

.diag-metric .lbl {
    font-size: 11px;
    color: var(--text-muted);

    text-transform: uppercase;
    margin-top: 4px;
}

/* ==========================================================
   PIPELINE PILLS
========================================================== */

.diag-pills {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
}

.diag-pill {
    background: var(--accent-dim);
    color: #d8d0ff;

    border-radius: 20px;

    padding: 5px 10px;

    font-size: 11px;
    font-weight: 600;
}

/* ==========================================================
   BUTTON
========================================================== */

.gr-button-primary {
    background: var(--accent) !important;
    border: none !important;

    color: white !important;

    border-radius: 10px !important;

    font-weight: 700 !important;
    font-size: 14px !important;

    padding: 12px !important;
}

.gr-button-primary:hover {
    opacity: 0.9 !important;
}

/* ==========================================================
   INPUT
========================================================== */

.gr-textbox textarea,
.gr-textbox input {
    background: var(--bg-input) !important;
    border: 1px solid var(--border) !important;

    color: var(--text-main) !important;

    border-radius: 10px !important;

    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
}

label span {
    color: var(--text-label) !important;
    font-weight: 600 !important;
}

/* ==========================================================
   IMAGE
========================================================== */

.gr-image {
    border: 1px solid var(--border) !important;
    border-radius: 10px !important;
    overflow: hidden !important;
}

/* ==========================================================
   FILE DOWNLOAD
========================================================== */

.gr-file {
    background: var(--bg-input) !important;
    border: 1px solid var(--border) !important;
    border-radius: 10px !important;
}

/* ==========================================================
   TABS
========================================================== */

button[role="tab"] {
    border-radius: 8px !important;
    font-weight: 600 !important;
}

button[role="tab"][aria-selected="true"] {
    background: var(--accent-dim) !important;
}

/* ==========================================================
   ACCORDION
========================================================== */

.gr-accordion {
    border-radius: 10px !important;
    overflow: hidden !important;
}

/* ==========================================================
   TRANSPARENT PREVIEW
========================================================== */

.transparent-preview {
    background-color: #f0f0f0 !important;
    background-image:
        linear-gradient(45deg, #d4d4d4 25%, transparent 25%),
        linear-gradient(-45deg, #d4d4d4 25%, transparent 25%),
        linear-gradient(45deg, transparent 75%, #d4d4d4 75%),
        linear-gradient(-45deg, transparent 75%, #d4d4d4 75%) !important;

    background-size: 20px 20px !important;

    background-position:
        0 0,
        0 10px,
        10px -10px,
        -10px 0 !important;
}

/* ==========================================================
   HILANGKAN PANAH
========================================================== */

.flow-arrow {
    display: none !important;
}

.download-list {
    padding: 0.25rem 0.5rem;
}

.download-list .gradio-container {
    gap: 0.15rem !important;
}

.download-list .file {
    margin-top: -0.35rem;
    margin-bottom: 0.4rem;
}
"""
MATRIX_PREVIEW_SIZE = 50



## Helper Functions

In [111]:
# ==========================================================
# HELPER: PREVIEW MATRIX KECIL (bukan matriks penuh)
# ==========================================================
def arr_to_preview_df(arr, size=MATRIX_PREVIEW_SIZE):
    if arr is None:
        return pd.DataFrame()
    h = min(size, arr.shape[0])
    w = min(size, arr.shape[1])
    if arr.ndim == 2:
        return pd.DataFrame(arr[:h, :w])
    elif arr.ndim == 3:
        return pd.DataFrame(arr[:h, :w, 0])
    return pd.DataFrame()


# ==========================================================
# HELPER: HISTOGRAM (Terisolasi dari State Global Matplotlib)
# ==========================================================
def make_histogram_image(arr, title="Histogram"):
    fig = Figure(figsize=(5, 2.4), dpi=100)
    canvas = FigureCanvasAgg(fig)
    fig.patch.set_facecolor("#1a1d27")

    ax = fig.add_subplot(111)
    ax.set_facecolor("#12151f")

    if arr.ndim == 2:
        ax.hist(arr.ravel(), bins=64, range=(0, 255), color="#7c6fcd", alpha=0.9)
    else:
        colors = ["#ff6b6b", "#6bcb77", "#4d96ff", "#f5c97a"]
        labels_full = ["R", "G", "B", "A"]
        n_ch = arr.shape[2]
        for c in range(n_ch):
            ax.hist(arr[:, :, c].ravel(), bins=64, range=(0, 255),
                     color=colors[c % 4], alpha=0.5, label=labels_full[c % 4])
        ax.legend(fontsize=7, facecolor="#1a1d27", labelcolor="#e8e8f0")

    ax.set_title(title, color="#e8e8f0", fontsize=10)
    ax.tick_params(colors="#a0a0c0", labelsize=8)
    for spine in ax.spines.values():
        spine.set_color("#2a2d3a")
    fig.tight_layout()

    canvas.draw()
    buf = np.asarray(canvas.buffer_rgba())
    img_rgb = cv2.cvtColor(buf, cv2.COLOR_RGBA2RGB)
    return img_rgb


# ==========================================================
# HELPER: STATISTIK PIKSEL
# ==========================================================
def compute_pixel_stats(arr, active_mask=None):
    h, w = arr.shape[0], arr.shape[1]
    total_px = h * w
    lines = []
    lines.append("Resolusi        : {} × {} px".format(w, h))

    if arr.ndim == 2:
        lines.append("Channel         : 1 (Grayscale/Biner)")
        lines.append("Min Pixel       : {}".format(int(arr.min())))
        lines.append("Max Pixel       : {}".format(int(arr.max())))
        lines.append("Mean            : {:.2f}".format(float(arr.mean())))
        lines.append("Std Deviation   : {:.2f}".format(float(arr.std())))
    else:
        n_ch = arr.shape[2]
        ch_names = ["R", "G", "B", "Alpha"] if n_ch == 4 else ["R", "G", "B"]
        lines.append("Channel         : {} ({})".format(n_ch, "RGBA" if n_ch == 4 else "RGB"))
        lines.append("Min Pixel       : {}".format(int(arr.min())))
        lines.append("Max Pixel       : {}".format(int(arr.max())))
        for i, name in enumerate(ch_names):
            lines.append("Mean {:<6}   : {:.2f}".format(name, float(arr[:, :, i].mean())))
        lines.append("Std Deviation   : {:.2f} (gabungan semua channel)".format(float(arr.std())))

    if active_mask is not None:
        active_count = int(np.sum(active_mask))
        bg_count = total_px - active_count
        pct = (active_count / total_px) * 100 if total_px > 0 else 0
        lines.append("Pixel Aktif     : {:,} px".format(active_count))
        lines.append("Pixel Background: {:,} px".format(bg_count))
        lines.append("Persentase Objek: {:.2f}%".format(pct))

    return "\n".join(lines)

# ==========================================================
# HELPER : DOWNLOAD
# ==========================================================
def build_download_links(
    png_file,
    transparent_file,
    svg_file,
    csv_file,
    zip_file
):
    html = "<ul>"

    if png_file:
        html += f'<li>PNG : <a href="/file={png_file}">Download</a></li>'

    if transparent_file:
        html += f'<li>PNG Alpha : <a href="/file={transparent_file}">Download</a></li>'

    if svg_file:
        html += f'<li>SVG : <a href="/file={svg_file}">Download</a></li>'

    if csv_file:
        html += f'<li>CSV : <a href="/file={csv_file}">Download</a></li>'

    if zip_file:
        html += f'<li>ZIP : <a href="/file={zip_file}">Download</a></li>'

    html += "</ul>"
    return html


# ==========================================================
# ANALISIS KUALITAS GAMBAR
# ==========================================================
def analyze_image_quality(gray, h, w):
    warnings = []
    suggestions = []

    mean_bright = np.mean(gray)
    if mean_bright < 80:
        warnings.append("Gambar terlalu gelap (kecerahan {:.0f}/255)".format(mean_bright))
        suggestions.append("Foto ulang di tempat lebih terang.")
    elif mean_bright > 230:
        warnings.append("Gambar terlalu terang/overexposed ({:.0f}/255)".format(mean_bright))
        suggestions.append("Kurangi cahaya saat memotret.")

    std_dev = np.std(gray)
    if std_dev < 20:
        warnings.append("Kontras sangat rendah (std {:.1f}) — kemungkinan pensil tipis.".format(std_dev))
        suggestions.append("Gunakan pulpen lebih tebal atau perbesar kontras.")

    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if lap_var < 50:
        warnings.append("Gambar buram/tidak fokus (sharpness {:.1f}).".format(lap_var))
        suggestions.append("Gunakan kamera yang lebih stabil dan fokus.")

    _, tmp = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark_ratio = np.sum(tmp == 0) / (h * w)
    if dark_ratio > 0.5:
        warnings.append("Lebih dari 50% area gelap — kemungkinan bayangan/background kotor.")
        suggestions.append("Foto di kertas putih bersih tanpa bayangan.")

    if w < 200 or h < 200:
        warnings.append("Resolusi terlalu kecil ({}×{} px).".format(w, h))
        suggestions.append("Upload foto minimal 400×400 px.")

    is_ok = len(warnings) == 0
    return warnings, suggestions, is_ok, mean_bright, std_dev, lap_var


def build_diagnosis_html(warnings, suggestions, is_ok, mean_bright, std_dev, lap_var, w, h, pipeline_steps):
    status_html = (
        '<div class="ok-box">✅ <strong>Gambar terdeteksi baik</strong> — siap diproses dengan pengaturan standar.</div>'
        if is_ok else
        '<div class="warn-box">⚠️ <strong>Beberapa hal perlu diperhatikan</strong> — sistem tetap menjalankan pipeline dengan penyesuaian otomatis.<ul style="margin:8px 0 0 16px;padding:0">'
        + "".join("<li style='margin-bottom:4px'>{} <span style='opacity:0.8'>{}</span></li>".format(w_, s_) for w_, s_ in zip(warnings, suggestions))
        + "</ul></div>"
    )

    metrics_html = """
    <div class="diag-grid">
      <div class="diag-metric"><div class="val">{:.0f}</div><div class="lbl">Kecerahan</div></div>
      <div class="diag-metric"><div class="val">{:.1f}</div><div class="lbl">Kontras (std)</div></div>
      <div class="diag-metric"><div class="val">{:.0f}</div><div class="lbl">Ketajaman</div></div>
      <div class="diag-metric"><div class="val">{}×{}</div><div class="lbl">Resolusi</div></div>
    </div>
    """.format(mean_bright, std_dev, lap_var, w, h)

    pills_html = ""
    if pipeline_steps:
        pills_html = '<div class="diag-pills">' + "".join(
            '<span class="diag-pill">✓ {}</span>'.format(p) for p in pipeline_steps
        ) + "</div>"

    return status_html + metrics_html + pills_html


def adaptive_preprocess(gray, std_dev, lap_var, mode, filter_method):
    processed = gray.copy()
    steps_used = []
    pipeline_flags = []

    if lap_var < 80:
        kernel_sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        processed = cv2.filter2D(processed, -1, kernel_sharp)
        steps_used.append("Sharpening (koreksi blur)")

    clahe_active = (mode == "Otomatis" and std_dev < 45)
    if clahe_active:
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        processed = clahe.apply(processed)
        steps_used.append("CLAHE (perkuat kontras lokal)")
        pipeline_flags.append("CLAHE aktif")

    if mode == "Manual":
        if filter_method == "Bilateral":
            processed = cv2.bilateralFilter(processed, d=7, sigmaColor=50, sigmaSpace=50)
            steps_used.append("Bilateral Filter (manual)")
            pipeline_flags.append("Bilateral Filter (manual)")
        elif filter_method == "Gaussian":
            processed = cv2.GaussianBlur(processed, (5, 5), 0)
            steps_used.append("Gaussian Blur (manual)")
            pipeline_flags.append("Gaussian Blur (manual)")
        else:
            steps_used.append("Tanpa filter noise (manual)")
    else:
        processed = cv2.bilateralFilter(processed, d=7, sigmaColor=50, sigmaSpace=50)
        steps_used.append("Bilateral Filter (redam noise, jaga tepi)")
        pipeline_flags.append("Bilateral Filter aktif")

    return processed, steps_used, pipeline_flags


def smart_threshold(preprocessed, h0, w0, mode, threshold_method):
    pipeline_flags = []
    total_px = h0 * w0

    otsu_val, thresh_otsu = cv2.threshold(
        preprocessed, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    if mode == "Manual":
        if threshold_method == "Otsu":
            pipeline_flags.append("Otsu Threshold (dipilih manual)")
            return thresh_otsu, "Otsu (manual)", otsu_val, pipeline_flags
        else:
            block_size = max(11, (min(h0, w0) // 20) | 1)
            thresh_adaptive = cv2.adaptiveThreshold(
                preprocessed, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                block_size,
                5
            )
            pipeline_flags.append("Adaptive Threshold (dipilih manual)")
            return thresh_adaptive, "Adaptive Gaussian (manual)", otsu_val, pipeline_flags

    active_ratio_otsu = np.sum(thresh_otsu == 0) / total_px
    otsu_bad = (active_ratio_otsu < 0.002) or (active_ratio_otsu > 0.35)

    if not otsu_bad:
        pipeline_flags.append("Otsu Threshold digunakan")
        return thresh_otsu, "Otsu", otsu_val, pipeline_flags

    block_size = max(11, (min(h0, w0) // 20) | 1)
    thresh_adaptive = cv2.adaptiveThreshold(
        preprocessed, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size,
        5
    )
    pipeline_flags.append("Adaptive Threshold aktif (fallback dari Otsu)")
    return thresh_adaptive, "Adaptive Gaussian", otsu_val, pipeline_flags


## Pipeline utama

In [112]:
SVG_QUALITY_EPSILON = {
    "Rendah (file kecil)": 2.0,
    "Sedang": 0.8,
    "Tinggi (detail halus)": 0.3,
}

# ==========================================================
# PIPELINE UTAMA
# ==========================================================
def process_signature(image, mode, threshold_method, filter_method, svg_quality):
    NOUT = 41
    if image is None:
        return (None,) * NOUT

    tmp_dir = tempfile.mkdtemp()
    epsilon = SVG_QUALITY_EPSILON.get(svg_quality, 0.8)

    img = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    h0, w0 = img.shape[:2]

    # ── ANALISIS KUALITAS ─────────────────────────────────
    gray_raw = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    std_dev  = np.std(gray_raw)
    lap_var  = cv2.Laplacian(gray_raw, cv2.CV_64F).var()

    warnings, suggestions, is_ok, mean_bright, std_dev2, lap_var2 = analyze_image_quality(gray_raw, h0, w0)

    # ── TAHAP 1: GRAYSCALE ────────────────────────────────
    gray = gray_raw.copy()
    gray_display = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    hist_gray = make_histogram_image(gray, "Distribusi Pixel — Grayscale")
    stats_gray = compute_pixel_stats(gray)
    df_gray = arr_to_preview_df(gray)
    info_gray = "Resolusi : {} × {} px\nChannel  : 1 (Grayscale)\nRata-rata: {:.0f}\nStd dev  : {:.1f}".format(w0, h0, np.mean(gray), std_dev)

    # ── TAHAP 2: PRE-PROCESSING + THRESHOLD ───────────────
    preprocessed, steps_used, pre_flags = adaptive_preprocess(gray, std_dev, lap_var, mode, filter_method)
    thresh, method_name, otsu_val, thresh_flags = smart_threshold(preprocessed, h0, w0, mode, threshold_method)

    pipeline_flags = pre_flags + thresh_flags
    diagnosis_html = build_diagnosis_html(warnings, suggestions, is_ok, mean_bright, std_dev, lap_var, w0, h0, pipeline_flags)

    thresh_display = cv2.cvtColor(thresh, cv2.COLOR_GRAY2RGB)
    hist_thresh = make_histogram_image(thresh, "Distribusi Pixel — Threshold")
    thresh_active_mask = (thresh == 255)
    stats_thresh = compute_pixel_stats(thresh, active_mask=thresh_active_mask)
    df_thresh = arr_to_preview_df(thresh)
    adaptasi_info = "\n".join(["• " + s for s in steps_used]) if steps_used else "• Tidak diperlukan"
    info_thresh = "Mode        : {}\nMetode      : {}\nOtsu (ref)  : {:.0f}\nPre-process :\n{}".format(
        mode, method_name, otsu_val, adaptasi_info
    )

    # ── TAHAP 3: MORFOLOGI ────────────────────────────────
    inv = cv2.bitwise_not(thresh)
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(inv, cv2.MORPH_OPEN, k_open, iterations=1)
    closed = opened.copy()

    morph_display = cv2.cvtColor(cv2.bitwise_not(closed), cv2.COLOR_GRAY2RGB)
    hist_morph = make_histogram_image(closed, "Distribusi Pixel — Morfologi")
    morph_active_mask = (closed == 255)
    stats_morph = compute_pixel_stats(closed, active_mask=morph_active_mask)
    df_morph = arr_to_preview_df(closed)
    noise_before = np.sum(inv == 255)
    noise_after = np.sum(closed == 255)
    info_morph = "Opening  : hapus noise kecil\nClosing  : nonaktif (jaga stroke tipis)\nKernel   : Ellipse 3×3\nSebelum  : {:,} px\nSesudah  : {:,} px\nDihapus  : {:,} px".format(
        noise_before, noise_after, max(0, noise_before - noise_after))

    # ── TAHAP 4: DETEKSI KONTUR ────────────────────────────
    contours, hierarchy = cv2.findContours(closed, cv2.RETR_TREE, cv2.CHAIN_APPROX_TC89_KCOS)
    total_px = h0 * w0
    min_area = max(6, total_px * 0.00005)
    valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]

    contour_vis = img.copy()
    cv2.drawContours(contour_vis, valid_contours, -1, (205, 111, 124), 2)
    contour_display = cv2.cvtColor(contour_vis, cv2.COLOR_BGR2RGB)
    hist_contour = make_histogram_image(contour_display, "Distribusi Pixel — Overlay Kontur")
    stats_contour = compute_pixel_stats(contour_display)
    df_contour = arr_to_preview_df(contour_display)
    info_contour = "Total kontur : {}\nKontur valid : {} (area > {:.1f}px)\nNoise dibuang: {}\nMetode approx: TC89_KCOS".format(
        len(contours), len(valid_contours), min_area, len(contours) - len(valid_contours))

    # ── TAHAP 5: CROP & RESIZE 400×400 ────────────────────
    coords = cv2.findNonZero(closed)
    if coords is None:
        empty_img = np.ones((400, 400, 3), dtype=np.uint8) * 30
        empty_hist = make_histogram_image(np.zeros((10, 10), dtype=np.uint8), "Tidak ada data")
        empty_df = pd.DataFrame()
        err = "Tidak ada objek terdeteksi.\nCoba upload ulang."
        return (
            diagnosis_html,
            gray_display, hist_gray, stats_gray, info_gray, df_gray,
            thresh_display, hist_thresh, stats_thresh, info_thresh, df_thresh,
            morph_display, hist_morph, stats_morph, info_morph, df_morph,
            contour_display, hist_contour, stats_contour, info_contour, df_contour,
            empty_img, empty_hist, err, err, empty_df,
            empty_img, empty_hist, err, err, empty_df,
            empty_img, empty_img, None, err,
            None, err,
            None, None, None, None,
        )

    x, y, w, h = cv2.boundingRect(coords)
    pad = 20
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(closed.shape[1], x + w + pad), min(closed.shape[0], y + h + pad)
    cropped = closed[y1:y2, x1:x2]

    TARGET = 400
    scale = min(TARGET / cropped.shape[1], TARGET / cropped.shape[0])
    nw, nh = int(cropped.shape[1] * scale), int(cropped.shape[0] * scale)
    resized = cv2.resize(cropped, (nw, nh), interpolation=cv2.INTER_AREA)
    _, resized = cv2.threshold(resized, 127, 255, cv2.THRESH_BINARY)

    canvas = np.zeros((TARGET, TARGET), dtype=np.uint8)
    ox, oy = (TARGET - nw) // 2, (TARGET - nh) // 2
    canvas[oy:oy+nh, ox:ox+nw] = resized

    final_png = cv2.bitwise_not(canvas)
    final_display = cv2.cvtColor(final_png, cv2.COLOR_GRAY2RGB)
    hist_final = make_histogram_image(final_png, "Distribusi Pixel — Crop & Resize")
    final_active_mask = (canvas == 255)
    stats_final = compute_pixel_stats(final_png, active_mask=final_active_mask)
    df_final = arr_to_preview_df(final_png)
    info_resize = "Bounding box : {}×{} px\nPadding      : {} px\nSkala resize : {:.3f}×\nKanvas akhir : {}×{} px".format(
        w, h, pad, scale, TARGET, TARGET)

    # ── TAHAP 5.5: BACKGROUND REMOVAL ──────────────────────
    # Konversi RGBA yang aman secara struktur untuk UI Web
    rgba = cv2.cvtColor(final_png, cv2.COLOR_GRAY2RGBA)
    rgba[final_png == 255] = [255, 255, 255, 0]
    rgba[final_png == 0] = [0, 0, 0, 255]

    hist_transparent = make_histogram_image(rgba, "Distribusi Pixel — RGBA")
    transparent_active_mask = (canvas == 255)
    stats_transparent = compute_pixel_stats(rgba, active_mask=transparent_active_mask)
    df_transparent = arr_to_preview_df(rgba)

    checker = np.zeros((TARGET, TARGET, 3), dtype=np.uint8)
    cell = 20
    for cy in range(0, TARGET, cell):
        for cx in range(0, TARGET, cell):
            color = 240 if ((cx // cell) + (cy // cell)) % 2 == 0 else 215
            checker[cy:cy+cell, cx:cx+cell] = (color, color, color)

    alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
    rgb_fg = rgba[:, :, :3].astype(np.float32)
    transparent_preview = (rgb_fg * alpha + checker.astype(np.float32) * (1 - alpha)).astype(np.uint8)

    transparent_pixel_count = int(np.sum(final_png == 0))
    info_transparent = "Area objek    : {:,} px\nArea transparan: {:,} px\nDimensi       : {}×{} px (RGBA)".format(
        transparent_pixel_count, TARGET*TARGET - transparent_pixel_count, TARGET, TARGET
    )

    # ── TAHAP 6: EXPORT SVG ─────────────────────────────────
    path_svg = os.path.join(tmp_dir, "signature_vectorized.svg")
    dwg = svgwrite.Drawing(path_svg, size=("400px", "400px"), profile="full")

    svg_contours, svg_hier = cv2.findContours(canvas, cv2.RETR_TREE, cv2.CHAIN_APPROX_TC89_KCOS)
    path_count = 0
    if svg_hier is not None:
        for cnt in svg_contours:
            if cv2.contourArea(cnt) > min_area:
                approx = cv2.approxPolyDP(cnt, epsilon=epsilon, closed=True)
                pts = [(int(p[0][0]), int(p[0][1])) for p in approx]
                if len(pts) >= 3:
                    dwg.add(dwg.polygon(points=pts, fill="black", stroke="black", stroke_width=0.3))
                    path_count += 1
    dwg.save()
    info_svg = "Format      : SVG\nKualitas    : {} (epsilon={})\nPath/polygon: {}\nViewbox     : 400×400".format(
        svg_quality, epsilon, path_count)

    # ── SIMPAN FILE ──────────────────────────────────────────
    path_clean = os.path.join(tmp_dir, "signature_clean.png")
    cv2.imwrite(path_clean, final_png)

    path_transparent = os.path.join(tmp_dir, "signature_transparent.png")
    # OpenCV imwrite membutuhkan format BGRA secara native untuk disave ke disk
    cv2.imwrite(path_transparent, cv2.cvtColor(rgba, cv2.COLOR_RGBA2BGRA))

    path_csv = os.path.join(tmp_dir, "signature_matrix.csv")
    pd.DataFrame(canvas).to_csv(path_csv, index=False)

    path_zip = os.path.join(tmp_dir, "hasil_signature.zip")
    with zipfile.ZipFile(path_zip, "w") as zf:
        zf.write(path_clean, arcname="signature_clean.png")
        zf.write(path_transparent, arcname="signature_transparent.png")
        zf.write(path_svg, arcname="signature_vectorized.svg")
        zf.write(path_csv, arcname="signature_matrix.csv")

    # ── OUTPUT FINAL (ringkasan akhir) ─────────────────────
    final_summary_stats = (
        "Jumlah kontur valid : {}\n"
        "Ukuran hasil        : {}×{} px\n"
        "Jumlah path SVG     : {}\n"
        "Kualitas SVG        : {}"
    ).format(len(valid_contours), TARGET, TARGET, path_count, svg_quality)

    download_html = f"""
    <p><b>PNG :</b> <a href="/file={path_clean}" target="_blank">Download</a></p>
    <p><b>PNG Alpha :</b> <a href="/file={path_transparent}" target="_blank">Download</a></p>
    <p><b>SVG :</b> <a href="/file={path_svg}" target="_blank">Download</a></p>
    <p><b>CSV :</b> <a href="/file={path_csv}" target="_blank">Download</a></p>
    <p><b>ZIP :</b> <a href="/file={path_zip}" target="_blank">Download</a></p>
    """
    return (
        diagnosis_html,
        gray_display,    hist_gray,        stats_gray,        info_gray,    df_gray,
        thresh_display,  hist_thresh,      stats_thresh,      info_thresh,  df_thresh,
        morph_display,   hist_morph,       stats_morph,       info_morph,   df_morph,
        contour_display, hist_contour,     stats_contour,     info_contour, df_contour,
        final_display,   hist_final,       stats_final,       info_resize,  df_final,
        transparent_preview, hist_transparent, stats_transparent, info_transparent, df_transparent,
        final_display, transparent_preview, path_svg, final_summary_stats,
        path_svg,
        info_svg,
        download_html,
    )



## UI Gradio

In [113]:
# ==========================================================
# UI
# ==========================================================
with gr.Blocks(css=CUSTOM_CSS, title="Signature to Vector") as demo:

    gr.HTML("""
    <div class="app-header">
        <h1>✍️ Mini Lab Pengolahan Citra Tanda Tangan</h1>
        <p>Upload → Diagnosis → Grayscale → Threshold → Morfologi → Kontur → Vectorization → SVG</p>
    </div>
    """)

    with gr.Tabs():

        # ==========================================================
        # TAB 1 · UPLOAD & OUTPUT
        # ==========================================================
        with gr.TabItem("Upload & Output"):

            gr.HTML('<div class="section-label">Upload + Setting</div>')

            with gr.Row(equal_height=False):
                with gr.Column(scale=4):
                    gr.HTML("""
                    <div class="card">
                      <div class="step-badge">📥 Input</div>
                      <div class="step-title">Input Gambar</div>
                      <div class="step-desc">Gunakan foto atau scan tanda tangan yang jelas.</div>
                    </div>
                    """)
                    input_image = gr.Image(type="numpy", label="Input Gambar")
                    img_info_box = gr.Textbox(
                        label="Info Gambar",
                        lines=2,
                        interactive=False,
                        placeholder="Ukuran gambar akan muncul di sini..."
                    )

                with gr.Column(scale=3):
                    gr.HTML("""
                    <div class="card">
                      <div class="step-badge">⚙️ Mode</div>
                      <div class="step-title">Mode & Setting Lanjutan</div>
                      <div class="step-desc">Mode otomatis cocok untuk mayoritas kasus. Mode manual memberi kontrol lebih detail.</div>
                    </div>
                    """)

                    mode_radio = gr.Radio(
                        choices=["Otomatis", "Manual"],
                        value="Otomatis",
                        label="Mode Pemrosesan"
                    )

                    with gr.Accordion("Setting Lanjutan", open=False):
                        threshold_method = gr.Dropdown(
                            choices=["Otsu", "Adaptive Gaussian"],
                            value="Otsu",
                            label="Threshold Method",
                            info="Hanya berlaku jika Mode = Manual"
                        )
                        filter_method = gr.Dropdown(
                            choices=["Bilateral", "Gaussian", "Tidak ada"],
                            value="Bilateral",
                            label="Filter Method",
                            info="Hanya berlaku jika Mode = Manual"
                        )
                        svg_quality = gr.Dropdown(
                            choices=["Rendah (file kecil)", "Sedang", "Tinggi (detail halus)"],
                            value="Sedang",
                            label="SVG Quality"
                        )

                    btn = gr.Button("▶ Jalankan Pipeline", variant="primary")

            gr.HTML('<div class="section-label">Hasil</div>')

            with gr.Row(equal_height=False):
                with gr.Column(scale=2):
                    gr.HTML("""
                    <div class="card">
                      <div class="step-badge">📦 Download</div>
                      <div class="step-title">Link Download</div>
                      <div class="step-desc">Unduh hasil dalam format PNG, PNG Alpha, SVG, CSV, dan ZIP.</div>
                    </div>
                    """)

                    # ==========================================================
                    # DOWNLOAD LIST
                    # ==========================================================
                    gr.HTML('<div class="section-label">Download</div>')

                    gr.HTML("""
                    <div class="card final-card">
                      <div class="step-badge">📦 Link Download</div>
                      <div class="step-desc">Klik file yang ingin diunduh.</div>
                    </div>
                    """)

                    download_links = gr.HTML(
                        value="""
                        <div style="opacity:0.6">
                            Belum ada file yang dapat diunduh.
                        </div>
                        """
                    )
                with gr.Column(scale=4):
                    gr.HTML("""
                    <div class="card final-card">
                      <div class="step-badge">✅ Output</div>
                      <div class="step-title">Output Final</div>
                      <div class="step-desc">Pratinjau hasil akhir dalam format PNG, PNG Transparan, dan SVG.</div>
                    </div>
                    """)
                    with gr.Tabs():
                        with gr.TabItem("PNG"):
                            final_png_preview = gr.Image(label="PNG Final", type="numpy", height=320)
                        with gr.TabItem("PNG Transparan"):
                            final_transparent_preview = gr.Image(
                                label="PNG Transparan",
                                type="numpy",
                                height=320,
                                elem_classes=["transparent-preview"]
                            )
                        with gr.TabItem("SVG"):
                            final_svg_preview = gr.File(label="File SVG Vektor", interactive=False)

            gr.HTML('<div class="section-label">Diagnosis Kualitas Gambar</div>')
            diagnosis_out = gr.HTML(
                value="<div class='warn-box' style='opacity:0.4'>Hasil diagnosis akan muncul di sini setelah gambar diproses...</div>"
            )


        # ==========================================================
        # TAB 2 · DETAIL PEMROSESAN
        # ==========================================================
        with gr.TabItem("Detail Pemrosesan"):

            gr.HTML('<div class="section-label">Tahap Pemrosesan</div>')

            def stage_panel(title, desc, img_label, is_transparent=False):
                gr.HTML(f"""
                <div class="card">
                  <div class="step-title">{title}</div>
                  <div class="step-desc">{desc}</div>
                </div>
                """)

                # Gambar utama
                img_comp = gr.Image(
                    label=img_label,
                    type="numpy",
                    height=300,
                    elem_classes=["transparent-preview"] if is_transparent else None
                )

                # Histogram + statistik sejajar
                with gr.Row():
                    with gr.Column(scale=2):
                        hist_comp = gr.Image(
                            label="Histogram Distribusi Pixel",
                            type="numpy",
                            height=220
                        )
                    with gr.Column(scale=1):
                        stats_comp = gr.Textbox(
                            label="Statistik Pixel",
                            lines=10,
                            interactive=False
                        )

                # Matriks di bawah
                df_comp = gr.DataFrame(
                    label=f"Preview Matrix {MATRIX_PREVIEW_SIZE}×{MATRIX_PREVIEW_SIZE} (pojok kiri-atas — unduh CSV untuk data lengkap)",
                    interactive=False
                )

                # Detail teknis tetap disimpan di bawah
                with gr.Accordion("Detail Teknis", open=False):
                    info_comp = gr.Textbox(
                        label="Informasi Proses",
                        lines=6,
                        interactive=False
                    )

                gr.HTML('<div class="flow-arrow">↓</div>')
                return img_comp, hist_comp, stats_comp, info_comp, df_comp

            with gr.Tabs():

                with gr.TabItem("Grayscale"):
                    (img_gray, hist_gray_comp, stats_gray_comp, info_gray, df_gray_comp) = stage_panel(
                        "Grayscale",
                        "Citra diubah ke 1-channel (0–255) agar threshold lebih akurat.",
                        "Hasil Grayscale"
                    )

                with gr.TabItem("Threshold"):
                    (img_thresh, hist_thresh_comp, stats_thresh_comp, info_thresh, df_thresh_comp) = stage_panel(
                        "Thresholding",
                        "Binarisasi citra — otomatis memilih metode terbaik atau sesuai pilihan manual.",
                        "Hasil Binarisasi"
                    )

                with gr.TabItem("Morphology"):
                    (img_morph, hist_morph_comp, stats_morph_comp, info_morph, df_morph_comp) = stage_panel(
                        "Morphology",
                        "Membersihkan noise kecil sambil menjaga goresan tipis tanda tangan.",
                        "Setelah Morfologi"
                    )

                with gr.TabItem("Contour"):
                    (img_contour, hist_contour_comp, stats_contour_comp, info_contour, df_contour_comp) = stage_panel(
                        "Contour Detection",
                        "Melacak batas setiap goresan dengan ambang area adaptif.",
                        "Overlay Kontur"
                    )

                with gr.TabItem("Vectorization"):
                    (img_final, hist_final_comp, stats_final_comp, info_final, df_final_comp) = stage_panel(
                        "Vectorization",
                        "Pemotongan area aktif dan penormalan kanvas sebelum ekspor vektor.",
                        "Hasil 400×400"
                    )

                with gr.TabItem("SVG Export"):
                    (img_transparent, hist_transparent_comp, stats_transparent_comp, info_transparent, df_transparent_comp) = stage_panel(
                        "SVG Export",
                        "Area putih dihapus menjadi transparan, lalu dipersiapkan untuk ekspor SVG.",
                        "Preview PNG Transparan",
                        is_transparent=True
                    )

                    gr.HTML("""
                    <div class="card">
                      <div class="step-badge">✒️ SVG</div>
                      <div class="step-title">Detail SVG Export</div>
                      <div class="step-desc">File SVG final dan ringkasan teknis ekspor.</div>
                    </div>
                    """)

                    # Jika ingin tampil lebih ringkas, biarkan label kecil saja
                    final_svg_preview = gr.File(label="SVG", interactive=False)
                    info_svg = gr.Textbox(label="Informasi SVG", lines=6, interactive=False)

    # ── Update info ukuran gambar saat upload ──
    def show_image_info(image):
        if image is None:
            return ""
        h, w = image.shape[:2]
        return f"Resolusi: {w} × {h} px\nChannel : {image.shape[2] if image.ndim == 3 else 1}"

    input_image.change(fn=show_image_info, inputs=[input_image], outputs=[img_info_box])

    # 41 Total Inputs and Outputs matched
    btn.click(
        fn=process_signature,
        inputs=[input_image, mode_radio, threshold_method, filter_method, svg_quality],
        outputs=[
            diagnosis_out,
            img_gray, hist_gray_comp, stats_gray_comp, info_gray, df_gray_comp,
            img_thresh, hist_thresh_comp, stats_thresh_comp, info_thresh, df_thresh_comp,
            img_morph, hist_morph_comp, stats_morph_comp, info_morph, df_morph_comp,
            img_contour, hist_contour_comp, stats_contour_comp, info_contour, df_contour_comp,
            img_final, hist_final_comp, stats_final_comp, info_final, df_final_comp,
            img_transparent, hist_transparent_comp, stats_transparent_comp, info_transparent, df_transparent_comp,
            final_png_preview, final_transparent_preview, final_svg_preview, final_stats_box,
            final_svg_preview,
            info_svg,

            download_links
        ]
    )


/tmp/ipykernel_11088/2336232094.py:4: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Signature to Vector") as demo:


## Launch

In [114]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26e24cf66684d4ed2f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
